# PerturbAI

In [4]:
import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

# throttle + retry so you don't trip the limit
con.sql("SET threads TO 2;")            # fewer files opened at once
con.sql("SET http_retries=5;")
con.sql("SET http_retry_wait_ms=2000;")
con.sql("SET http_retry_backoff=2;")

In [13]:
import duckdb
duckdb.sql("INSTALL httpfs; LOAD httpfs;")
duckdb.sql("""
  SELECT gene_target, count(*) AS n
  FROM 'hf://datasets/perturbai/wholebrain_crispr_atlas/metadata/all_obs.parquet'
  WHERE gene_target LIKE '%Taf1%'
  GROUP BY gene_target
""").show()

┌──────────────────────────────────────────┬───────┐
│               gene_target                │   n   │
│                 varchar                  │ int64 │
├──────────────────────────────────────────┼───────┤
│ Hras|Lgi1|Taf1b                          │     1 │
│ Taf1b                                    │  1146 │
│ Cfap69|Cnr1|Prr12|Setdb1|Taf1|Uchl1      │     1 │
│ Lztr1|Taf1                               │     1 │
│ Ahr|Taf1                                 │     1 │
│ Runx3|Taf1                               │     1 │
│ Nr1h3|Taf1b                              │     3 │
│ Camk2b|Dach1|Itga9|Taf1b|Trim2           │     1 │
│ Ppard|Taf1                               │     1 │
│ Sox17|Taf1b|Tcf7l2                       │     1 │
│      ·                                   │     · │
│      ·                                   │     · │
│      ·                                   │     · │
│ Pitx2|Taf1b                              │     1 │
│ Ddr2|Taf1                                │  

### Number of cells for eahc cell type

In [5]:
df = con.sql("""
  SELECT predicted_subclass, count(*) AS n
  FROM 'hf://datasets/perturbai/wholebrain_crispr_atlas/metadata/all_obs.parquet'
  WHERE gene_target = 'Taf1'
    AND neuron_type <> 'Non-Neuron'
    AND passes_qc = true
  GROUP BY predicted_subclass
  ORDER BY n DESC
""").to_df()#show()

df

,predicted_subclass,n
0,052 Pvalb Gaba,51
1,006 L4/5 IT CTX Glut,49
2,151 TH Prkcd Grin2c Glut,44
3,007 L2/3 IT CTX Glut,39
4,049 Lamp5 Gaba,33
...,...,...
107,143 MM-ant Foxb1 Glut,1
108,050 Lamp5 Lhx6 Gaba,1
109,285 MY Lhx1 Gly-Gaba,1
110,199 PAG-MRN-RN Foxa2 Gaba,1


In [9]:

df[df.predicted_subclass.str.contains("STR D")].sort_values("n", ascending=False)



,predicted_subclass,n
13,061 STR D1 Gaba,15
19,062 STR D2 Gaba,10
101,063 STR D1 Sema5a Gaba,1


### Download expression data

In [14]:
con.sql("""
  COPY (
    SELECT *
    FROM 'hf://datasets/perturbai/wholebrain_crispr_atlas/data/*.parquet'
    WHERE gene_target = 'Taf1'
      AND predicted_subclass LIKE '%STR D%'
      AND passes_qc = true
  ) TO 'taf1_strd.parquet' (FORMAT parquet);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
n_ctrl = con.sql("""
  SELECT count(*) FROM 'hf://datasets/perturbai/wholebrain_crispr_atlas/metadata/all_obs.parquet'
  WHERE gene_target = 'Non_target' AND predicted_subclass LIKE '%STR D%' AND passes_qc = true
""").fetchone()[0]
print(n_ctrl, "control cells available")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

6714 control cells available


In [6]:
frac = min(1.0, 1000 / n_ctrl)
con.sql(f"""
  COPY (
    SELECT *
    FROM 'hf://datasets/perturbai/wholebrain_crispr_atlas/data/*.parquet'
    WHERE gene_target = 'Non_target'
      AND predicted_subclass LIKE '%STR D%'
      AND passes_qc = true
      AND random() < {frac}
  ) TO 'nt_strd.parquet' (FORMAT parquet);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
df = con.sql("SELECT * FROM '/home/gdallagl/myworkdir/XDP/data/_old/taf1_strd.parquet'").df()
df

,genes,expressions,cell_id,batch,scp_name,source,sex,sample_label,num_rna_umi,num_genes,...,predicted_supertype,predicted_supertype_probability,predicted_cluster,predicted_cluster_probability,neuron_type,neighborhood,region_level1,region_level2,cluster,passes_qc
0,"[4, 6, 9, 13, 18, 24, 25, 27, 28, 31, 32, 33, ...","[2, 6, 1, 1, 1, 3, 3, 2, 1, 4, 4, 4, 1, 2, 3, ...",GCAGTCGCAGGATTGCCTCAGCCATC-1:SCP039,WB8588_1,WB8588_1_2,mouse3,M,3L,16454.0,5959,...,0281 STR D1 Sema5a Gaba_1,1.00,0990 STR D1 Sema5a Gaba_1,1.00,GABA,Subpallium-GABA,Striatum,STRd,51,True
1,"[0, 5, 6, 7, 12, 13, 19, 23, 24, 25, 33, 39, 4...","[1, 2, 1, 1, 4, 1, 2, 1, 2, 2, 1, 2, 1, 4, 3, ...",CAGTACCTCATACAGAACAAGCGGAT-1:SCP066,WB8588_2,WB8588_2_1,mouse9,M,9R,11235.0,4979,...,0267 STR D1 Gaba_3,1.00,0950 STR D1 Gaba_3,1.00,GABA,Subpallium-GABA,Striatum,STRd,8,True
2,"[4, 5, 6, 9, 12, 13, 14, 24, 28, 31, 32, 33, 3...","[1, 1, 5, 2, 1, 1, 1, 2, 1, 4, 1, 1, 1, 3, 2, ...",CATAAGGGTTGTGCGCTGTTGGTAAG-1:SCP066,WB8588_2,WB8588_2_1,mouse7,F,7R,9712.0,4541,...,0270 STR D1 Gaba_6,1.00,0959 STR D1 Gaba_6,0.79,GABA,Subpallium-GABA,Striatum,STRv,8,True
3,"[0, 5, 6, 7, 9, 12, 23, 24, 25, 28, 31, 41, 42...","[2, 1, 2, 3, 3, 1, 1, 1, 1, 3, 1, 1, 3, 1, 1, ...",GGTAACGTCACATGCCGAGAGGGTCT-1:SCP066,WB8588_2,WB8588_2_1,mouse22,M,22R,7553.0,3954,...,0267 STR D1 Gaba_3,0.90,0950 STR D1 Gaba_3,0.90,GABA,Subpallium-GABA,Striatum,STRd,8,True
4,"[0, 5, 6, 7, 12, 13, 19, 23, 24, 27, 28, 31, 3...","[1, 2, 4, 2, 1, 1, 2, 1, 3, 1, 1, 2, 1, 1, 2, ...",GCAAGGGAGGTCTACCAGCCTGTCCT-1:SCP068,WB8588_2,WB8588_2_3,mouse15,F,15R,9490.0,4626,...,0270 STR D1 Gaba_6,1.00,0959 STR D1 Gaba_6,0.97,GABA,Subpallium-GABA,Striatum,STRv,8,True
5,"[0, 3, 6, 7, 9, 12, 18, 19, 28, 38, 42, 47, 68...","[3, 1, 1, 2, 1, 1, 4, 1, 2, 1, 4, 1, 1, 1, 1, ...",AGGCGTGGTATCCTAAGTGCAATGAC-1:SCP068,WB8588_2,WB8588_2_3,mouse13,F,13R,5803.0,3394,...,0267 STR D1 Gaba_3,0.58,0950 STR D1 Gaba_3,0.58,GABA,Subpallium-GABA,Striatum,STRd,8,True
6,"[0, 4, 6, 9, 12, 13, 19, 20, 23, 24, 25, 31, 3...","[1, 3, 8, 3, 8, 1, 3, 1, 2, 2, 3, 3, 1, 2, 1, ...",ACATATTGTACCTCTCCAAGGTGTTC-1:SCP069,WB8588_3,WB8588_3_1,mouse52,M,52L,17207.0,6163,...,0272 STR D1 Gaba_8,0.75,0961 STR D1 Gaba_8,0.75,GABA,Subpallium-GABA,Striatum,STRd,8,True
7,"[5, 6, 7, 12, 20, 23, 24, 25, 31, 35, 42, 44, ...","[3, 6, 2, 1, 1, 1, 1, 5, 3, 1, 2, 2, 1, 1, 2, ...",GTCGCGCTCTTCGCAAGTGCCCGATA-1:SCP070,WB8588_3,WB8588_3_2,mouse34,F,34L,7121.0,3812,...,0268 STR D1 Gaba_4,0.65,0953 STR D1 Gaba_4,0.63,GABA,Subpallium-GABA,Striatum,STRd,8,True
8,"[4, 6, 12, 19, 24, 25, 27, 31, 33, 38, 41, 42,...","[1, 1, 3, 1, 3, 2, 2, 3, 2, 1, 2, 4, 1, 1, 1, ...",TAATCGAAGGCCATTCAGCTGGCTAC-1:SCP070,WB8588_3,WB8588_3_2,mouse52,M,52L,5261.0,3172,...,0274 STR D2 Gaba_1,0.58,0970 STR D2 Gaba_1,0.57,GABA,Subpallium-GABA,Striatum,STRd,8,True
9,"[0, 4, 5, 6, 9, 12, 23, 24, 25, 28, 31, 33, 36...","[2, 2, 1, 4, 3, 2, 2, 1, 1, 5, 1, 1, 1, 2, 1, ...",TGAACACAGTTGCATTAGCAACATAC-1:SCP070,WB8588_3,WB8588_3_2,mouse38,M,38R,10119.0,4629,...,0270 STR D1 Gaba_6,1.00,0959 STR D1 Gaba_6,1.00,GABA,Subpallium-GABA,Striatum,STRv,8,True


In [27]:
df[(df.gene_target == "TAF1")]
df[(df.group_name.str.contains("STR D"))]


,deg_names,scores,logfoldchanges,pvals,pvals_adj,group_name,gene_target,control_label,n_pert_matched,n_ctrl_matched


In [36]:
df.group_name.unique()
df[df.gene_target.str.contains("Taf1")]

,deg_names,scores,logfoldchanges,pvals,pvals_adj,group_name,gene_target,control_label,n_pert_matched,n_ctrl_matched


In [ ]:
# is DEG in taf1 alredy done?

import pandas as pd
df = pd.read_excel("/home/gdallagl/myworkdir/XDP/data/_old/media-6.xlsx", skiprows=1)   # the S6A sheet

# 1) what do the striatal cell-type bins look like here?
print([g for g in df.group_name.unique() if "STR" in g.upper()])

# 2) Taf1 only (NOT Taf1b), striatal bins
taf1 = df[(df.gene_target == "Taf1") & (df.group_name.str.contains("STR"))].copy()

# 3) power check — how many Taf1 cells backed each cell type
print(taf1[["group_name", "n_pert_matched"]].drop_duplicates())

# 4) the actual hit list at the paper's real threshold
sig = (taf1[taf1.pvals_adj < 0.05]
       .reindex(taf1.pvals_adj.abs().sort_values().index))  # or sort by |scores|
sig = taf1[taf1.pvals_adj < 0.05].sort_values("scores", key=lambda s: s.abs(), ascending=False)
sig[["deg_names", "logfoldchanges", "pvals_adj", "group_name", "n_pert_matched"]]

[]
Empty DataFrame
Columns: [group_name, n_pert_matched]
Index: []


,deg_names,logfoldchanges,pvals_adj,group_name,n_pert_matched


In [12]:
import numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad, scanpy as sc

# --- gene axis: index in `genes` column maps to row order here ---
gm = con.sql("""
  SELECT * FROM 'hf://datasets/perturbai/wholebrain_crispr_atlas/metadata/gene_metadata.parquet'
""").df()
# pick the symbol column (adjust if it's named differently after you inspect gm.columns)
name_col = next(c for c in ['gene_name','gene_symbol','symbol','name'] if c in gm.columns)
var_names = gm[name_col].astype(str).to_numpy()
n_genes = len(var_names)

def to_anndata(pdf):
    lengths = pdf['genes'].map(len).to_numpy()
    indptr  = np.zeros(len(pdf)+1, dtype=np.int64); indptr[1:] = np.cumsum(lengths)
    indices = np.concatenate(pdf['genes'].to_numpy()).astype(np.int64)
    data    = np.concatenate(pdf['expressions'].to_numpy()).astype(np.float32)
    assert indices.max() < n_genes, "gene index out of range — indexing assumption is wrong"
    X = sp.csr_matrix((data, indices, indptr), shape=(len(pdf), n_genes))
    obs = pdf.drop(columns=['genes','expressions']).reset_index(drop=True)
    a = ad.AnnData(X=X, obs=obs); a.var_names = var_names
    return a

taf1 = con.sql("SELECT * FROM 'taf1_strd.parquet'").df()
nt   = con.sql("SELECT * FROM 'nt_strd.parquet'").df()
adata = to_anndata(pd.concat([taf1, nt], ignore_index=True))

# --- normalize (expressions are raw counts) then Wilcoxon Taf1 vs Non_target ---
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs['gene_target'] = adata.obs['gene_target'].astype('category')

n_taf1 = int((adata.obs.gene_target == 'Taf1').sum())
if n_taf1 < 30:
    print(f"⚠️ only {n_taf1} Taf1 cells — results are exploratory, not reliable")

sc.tl.rank_genes_groups(adata, 'gene_target',
                        groups=['Taf1'], reference='Non_target', method='wilcoxon')
res = sc.get.rank_genes_groups_df(adata, group='Taf1').sort_values('scores', key=lambda s: s.abs(), ascending=False)
res_sig #= res[res.pvals_adj < 0.05].sort_values('scores', key=lambda s: s.abs(), ascending=False)
res_sig
res

/home/gdallagl/.pyenv/versions/3.11.8/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


⚠️ only 26 Taf1 cells — results are exploratory, not reliable


,names,scores,logfoldchanges,pvals,pvals_adj
0,Ncan,3.824613,1.009923,0.000131,1.0
1,Arl3,3.685954,1.049500,0.000228,1.0
2,Lifr,3.417091,1.652463,0.000633,1.0
3,Pcca,3.233791,0.899114,0.001222,1.0
4,Lamp2,3.219587,0.908511,0.001284,1.0
...,...,...,...,...,...
7658,Speer4e,0.000000,0.000000,1.000000,1.0
7657,Cd209f,0.000000,0.000000,1.000000,1.0
7656,Cd209c,0.000000,0.000000,1.000000,1.0
7655,Cd209b,0.000000,0.000000,1.000000,1.0
